In [1]:
import os, time, math, json
from datetime import datetime, timedelta, timezone
import sys
sys.path.append("../")
import random
from dotenv import load_dotenv
from qdrant_client import QdrantClient
import numpy as np
import polars as pl
from common.db import Connection
import polars as pl
from boto3.dynamodb.conditions import Attr
from botocore.exceptions import ClientError, EndpointConnectionError
from decimal import Decimal

load_dotenv()

conn = Connection()
d = conn.get_dynamo()
dynamo = d.Table("event")

qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_ENDPOINT"),
    api_key=os.getenv("QDRANT_API")
)

In [2]:
def now_utc_ms() -> int:
    return int(datetime.now(timezone.utc).timestamp() * 1000)

def days_ago_ms(days: int) -> int:
    return int((datetime.now(timezone.utc) - timedelta(days=days)).timestamp() * 1000)

def _to_py(v):
    """DynamoDB Decimal/중첩 구조를 파이썬 기본 타입으로 정규화"""
    if isinstance(v, Decimal):
        # 소수점이 없으면 int, 있으면 float로
        if v == v.to_integral_value():
            return int(v)
        return float(v)
    if isinstance(v, dict):
        return {k: _to_py(x) for k, x in v.items()}
    if isinstance(v, list):
        return [_to_py(x) for x in v]
    return v

def _ensure_keys(d, keys):
    for k in keys:
        if k not in d:
            d[k] = None
    return d

In [10]:
def fetch_logs_last_ndays(
    days: int = 7,
    max_items_per_page: int = 1000,
    projection = ("member_id", "event_type", "target_type", "target_id", "timestamp"),
    consistent_read: bool = False,
    throttle_backoff_sec: float = 0.5,
    throttle_backoff_max_sec: float = 8.0,
) -> pl.DataFrame:
    """
    최근 N일 로그를 DynamoDB에서 스캔하여 Polars DataFrame으로 반환합니다.

    Args:
        days: 최근 N일
        max_items_per_page: Scan 페이지 당 아이템 수 (최대 1,000 권장)
        projection: 가져올 컬럼 목록
        consistent_read: 강한 일관성 읽기 사용 여부 (비용↑/지연↑)
        throttle_backoff_sec: 스로틀링 초기 백오프
        throttle_backoff_max_sec: 스로틀링 최대 백오프

    Returns:
        Polars DataFrame (컬럼: member_id, event_type, target_type, target_id, timestamp)
    """
    start_ms = days_ago_ms(days)
    end_ms = now_utc_ms()

    # timestamp는 예약어 충돌 우려가 있으므로 별칭 사용
    expr_attr_names = {"#ts": "timestamp"}
    projection_expr_parts = []
    for col in projection:
        if col == "timestamp":
            projection_expr_parts.append("#ts")
        else:
            projection_expr_parts.append(col)
    projection_expression = ", ".join(projection_expr_parts)

    filter_expression = Attr("timestamp").between(start_ms, end_ms)

    items: list[dict] = []
    last_evaluated_key: dict | None = None

    while True:
        try:
            scan_kwargs = {
                "FilterExpression": filter_expression,
                "ProjectionExpression": projection_expression,
                "ExpressionAttributeNames": expr_attr_names,
                "Limit": max_items_per_page,
                "ConsistentRead": consistent_read,
            }
            if last_evaluated_key:                 # ✅ 키가 있을 때만 전달
                scan_kwargs["ExclusiveStartKey"] = last_evaluated_key

            resp = dynamo.scan(**scan_kwargs)

        except (ClientError, EndpointConnectionError) as e:
            if isinstance(e, ClientError) and e.response.get("Error", {}).get("Code") in {
                "ProvisionedThroughputExceededException",
                "ThrottlingException",
                "RequestLimitExceeded",
            }:
                time.sleep(throttle_backoff_sec)
                throttle_backoff_sec = min(throttle_backoff_sec * 2, throttle_backoff_max_sec)
                continue
            raise

        page_items = resp.get("Items", [])
        if page_items:
            items.extend(page_items)

        last_evaluated_key = resp.get("LastEvaluatedKey")
        if not last_evaluated_key:
            break

    # 빈 결과 처리
    if not items:
        return pl.DataFrame(
            schema={
                "member_id": pl.Int64,
                "event_type": pl.Utf8,
                "target_type": pl.Utf8,
                "target_id": pl.Utf8,
                "timestamp": pl.Int64,
            }
        )


    # 안전한 캐스팅 (DynamoDB 숫자/문자 혼재 가능)
    wanted_cols = ["member_id", "event_type", "target_type", "target_id", "timestamp"]
    items_norm = [_ensure_keys(_to_py(it), wanted_cols) for it in items]

    df = pl.from_dicts(
        items_norm,
        schema={
            "member_id": pl.Int64,
            "event_type": pl.Utf8,
            "target_type": pl.Utf8,
            "target_id": pl.Utf8,
            "timestamp": pl.Int64,  # ms epoch
        },
    ).with_columns(
        pl.col("member_id").cast(pl.Int64, strict=False),
        pl.col("event_type").cast(pl.Utf8, strict=False),
        pl.col("target_type").cast(pl.Utf8, strict=False),
        pl.col("target_id").cast(pl.Utf8, strict=False),
        pl.col("timestamp").cast(pl.Int64, strict=False),
    )

    df = df.with_columns(
        pl.from_epoch(pl.col("timestamp"), "ms").dt.replace_time_zone("UTC").alias("datetime_utc")
    )
    # UTC → KST 변환
    df = df.with_columns(
        pl.col("datetime_utc").dt.convert_time_zone("Asia/Seoul").alias("datetime_kst")
    )
        

    # 안전 필터 (범위 밖 데이터 제거)
    df = df.filter(
        pl.col("timestamp").is_not_null()
        & (pl.col("timestamp") >= start_ms)
        & (pl.col("timestamp") <= end_ms)
    )

    return df

In [11]:
df = fetch_logs_last_ndays(
    days=7,
    max_items_per_page=1000,
    projection=("member_id", "event_type", "target_type", "target_id", "timestamp"),
    consistent_read=False,  # EDA면 보통 False로 충분
)

In [12]:
df

member_id,event_type,target_type,target_id,timestamp,datetime_utc,datetime_kst
i64,str,str,str,i64,"datetime[ms, UTC]","datetime[ms, Asia/Seoul]"
184,"""f_imp""","""article""","""5eAcUT3huFfSeDIaLNsEk1aUaAu""",1755261379796,2025-08-15 12:36:19.796 UTC,2025-08-15 21:36:19.796 KST
184,"""f_imp""","""article""","""5eAcUT3huFfSeDIaLNsEk1aUaAu""",1755261380395,2025-08-15 12:36:20.395 UTC,2025-08-15 21:36:20.395 KST
184,"""f_imp""","""article""","""c9MIUl1Gf7VMYjbWOlaRkf9P9nK""",1755261381407,2025-08-15 12:36:21.407 UTC,2025-08-15 21:36:21.407 KST
184,"""f_imp""","""article""","""c9MIUl1Gf7VMYjbWOlaRkf9P9nK""",1755261382256,2025-08-15 12:36:22.256 UTC,2025-08-15 21:36:22.256 KST
184,"""f_imp""","""article""","""hu2bqF1YlULJByQ8rv49nZRC5El""",1755261383115,2025-08-15 12:36:23.115 UTC,2025-08-15 21:36:23.115 KST
…,…,…,…,…,…,…
182,"""f_imp""","""article""","""8K9wnURboXGrG73rQCt9sZFsNvm""",1755253220479,2025-08-15 10:20:20.479 UTC,2025-08-15 19:20:20.479 KST
182,"""like""","""article""","""8K9wnURboXGrG73rQCt9sZFsNvm""",1755253228583,2025-08-15 10:20:28.583 UTC,2025-08-15 19:20:28.583 KST
182,"""share""","""article""","""8K9wnURboXGrG73rQCt9sZFsNvm""",1755253232135,2025-08-15 10:20:32.135 UTC,2025-08-15 19:20:32.135 KST


# User-Centric EDA 함수

In [45]:
def analyze_user_logs(df: pl.DataFrame, total_user_count: int | None = None) -> dict:
    """
    유저 행동 로그 분석 (최근 N일 데이터 기반)

    Args:
        df: Polars DataFrame (컬럼: member_id, event_type, target_type, target_id, timestamp)
        total_user_count: 전체 유저 수 (없으면 cold-start 비율은 None)

    Returns:
        dict: 분석 결과 (집계 테이블 및 지표)
    """
    results = {}

    # ------------------------------
    # (1) 활동 유저 수
    # ------------------------------
    active_users = df.select("member_id").n_unique()
    if total_user_count:
        cold_start_ratio = 1 - (active_users / total_user_count)
    else:
        cold_start_ratio = None

    results["active_users"] = active_users
    results["cold_start_ratio"] = cold_start_ratio

    # ------------------------------
    # (2) 유저별 이벤트 수 분포
    # ------------------------------
    user_event_counts = (
        df.group_by(["member_id", "event_type"])
          .agg(pl.len().alias("cnt"))
          .pivot(values="cnt", index="member_id", on="event_type")
          .fill_null(0)
    )

    # 유저별 전체 이벤트 수
    user_event_counts = user_event_counts.with_columns(
        total_events = pl.sum_horizontal(pl.all().exclude("member_id"))
    )

    # 전체 이벤트 분포 요약
    total_events_desc = user_event_counts["total_events"].describe()
    print(total_events_desc)

    # Heavy User: 상위 10% 기준
    threshold = user_event_counts["total_events"].quantile(0.9, interpolation="higher")
    heavy_user_ratio = (user_event_counts["total_events"] >= threshold).mean()

    results["user_event_counts"] = user_event_counts
    results["total_events_desc"] = total_events_desc
    results["heavy_user_ratio"] = heavy_user_ratio

    # ------------------------------
    # (3) 행동 다양성
    # ------------------------------
    # 몇 가지 event_type을 했는지 (e.g. f_imp만 했는지, like까지 했는지)
    user_event_counts = user_event_counts.with_columns(
        diversity = (pl.sum_horizontal((user_event_counts.drop("member_id") > 0)))
    )

    diversity_dist = (
        user_event_counts.group_by("diversity")
                         .agg(pl.len().alias("user_count"))
                         .sort("diversity")
    )
    results["diversity_dist"] = diversity_dist

    # ------------------------------
    # (4) 전환 퍼널  — 시간 순/같은 유저·아이템 기준
    # ------------------------------
    def _compute_time_ordered_funnel(df: pl.DataFrame, within_hours: int | None = None):
        """
        시간 순서를 고려한 퍼널을 계산합니다.
        - 단위: (member_id, target_id)
        - 규칙:
        saw: f_imp 발생
        clicked: saw 이후 article_in 발생 (옵션: within_hours 제한)
        liked: clicked 이후 like 발생
        archived: clicked 이후 archive 발생
        """
        # article 만 퍼널 대상 (blog_in 등 제외)
        df_art = df.filter(pl.col("target_type") == "article")

        # 유저·아이템 단위로 이벤트별 '최초 시각' 집계
        ui = (
            df_art.group_by(["member_id", "target_id"])
            .agg([
                pl.when(pl.col("event_type") == "f_imp").then(pl.col("timestamp")).min().alias("ts_imp"),
                pl.when(pl.col("event_type") == "article_in").then(pl.col("timestamp")).min().alias("ts_click"),
                pl.when(pl.col("event_type") == "like").then(pl.col("timestamp")).min().alias("ts_like"),
                pl.when(pl.col("event_type") == "archive").then(pl.col("timestamp")).min().alias("ts_archive"),
            ])
        )

        # 시간 제약(ms)
        window_ms = None if within_hours is None else within_hours * 60 * 60 * 1000

        # 단계별 충족 여부 (시간 순서 보장)
        ui = ui.with_columns(
            saw = pl.col("ts_imp").is_not_null(),
            clicked = (
                pl.col("ts_imp").is_not_null()
                & pl.col("ts_click").is_not_null()
                & (pl.col("ts_click") >= pl.col("ts_imp"))
                & (
                    True if window_ms is None
                    else (pl.col("ts_click") <= pl.col("ts_imp") + window_ms)
                )
            ),
        ).with_columns(
            liked = (
                pl.col("clicked")
                & pl.col("ts_like").is_not_null()
                & (pl.col("ts_like") >= pl.col("ts_click"))
            ),
            archived = (
                pl.col("clicked")
                & pl.col("ts_archive").is_not_null()
                & (pl.col("ts_archive") >= pl.col("ts_click"))
            ),
        )

        # 카운트 & 비율
        saw_cnt = ui.filter(pl.col("saw")).height
        click_cnt = ui.filter(pl.col("clicked")).height
        like_cnt = ui.filter(pl.col("liked")).height
        archive_cnt = ui.filter(pl.col("archived")).height

        funnel_rates = {
            "CTR (clicked / saw) %": round((click_cnt / saw_cnt)*100, 3) if saw_cnt else None,
            "Like rate (liked / clicked) %": round((like_cnt / click_cnt)*100, 3) if click_cnt else None,
            "Archive rate (archived / clicked) %": round((archive_cnt / click_cnt)*100, 3) if click_cnt else None,
        }

        return ui, {
            "saw_cnt": saw_cnt,
            "click_cnt": click_cnt,
            "like_cnt": like_cnt,
            "archive_cnt": archive_cnt,
            "funnel_rates": funnel_rates,
        }

    # within_hours=None 이면 시간 제한 없이, 예: within_hours=24 로 두면 '노출 24시간 내 클릭' 기준
    ui_level, funnel_summary = _compute_time_ordered_funnel(df, within_hours=2)

    results["funnel_ui_level"] = ui_level  # (member_id, target_id) 단위의 단계별 bool 컬럼 포함 DF
    results["funnel_summary"] = funnel_summary

    return results

In [46]:
total_user = int(conn.execute('select count(distinct member_id) as cnt from member')['cnt'][0])
results = analyze_user_logs(df, total_user)

shape: (9, 2)
┌────────────┬────────────┐
│ statistic  ┆ value      │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ count      ┆ 33.0       │
│ null_count ┆ 0.0        │
│ mean       ┆ 63.454545  │
│ std        ┆ 151.199804 │
│ min        ┆ 1.0        │
│ 25%        ┆ 2.0        │
│ 50%        ┆ 13.0       │
│ 75%        ┆ 52.0       │
│ max        ┆ 833.0      │
└────────────┴────────────┘


In [43]:
print("=== (1) 활동 유저 수 ===")
print(f"최근 7일 활동 유저: {results['active_users']}")
if results["cold_start_ratio"] is not None:
    print(f"Cold-start 유저 비율: {results['cold_start_ratio']:.2%}")

print("\n=== (2) 이벤트 수 분포 요약 ===")
print(results["total_events_desc"])
print(f"Heavy user 비율 (상위10%): {results['heavy_user_ratio']:.2%}")

print("\n=== (3) 행동 다양성 분포 ===")
print(results["diversity_dist"])

print("=== (4) 전환 퍼널 (시간 순/동일 아이템 기준) ===")
print(f"노출 쌍 수: {results['funnel_summary']['saw_cnt']}")
print(f"클릭 쌍 수: {results['funnel_summary']['click_cnt']}")
print(f"좋아요 쌍 수: {results['funnel_summary']['like_cnt']}")
print(f"아카이브 쌍 수: {results['funnel_summary']['archive_cnt']}")
print("전환율:", results['funnel_summary']['funnel_rates'])

=== (1) 활동 유저 수 ===
최근 7일 활동 유저: 33
Cold-start 유저 비율: 81.77%

=== (2) 이벤트 수 분포 요약 ===
shape: (9, 2)
┌────────────┬────────────┐
│ statistic  ┆ value      │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ count      ┆ 33.0       │
│ null_count ┆ 0.0        │
│ mean       ┆ 63.454545  │
│ std        ┆ 151.199804 │
│ min        ┆ 1.0        │
│ 25%        ┆ 2.0        │
│ 50%        ┆ 13.0       │
│ 75%        ┆ 52.0       │
│ max        ┆ 833.0      │
└────────────┴────────────┘
Heavy user 비율 (상위10%): 12.12%

=== (3) 행동 다양성 분포 ===
shape: (7, 2)
┌───────────┬────────────┐
│ diversity ┆ user_count │
│ ---       ┆ ---        │
│ u32       ┆ u32        │
╞═══════════╪════════════╡
│ 2         ┆ 12         │
│ 3         ┆ 6          │
│ 4         ┆ 5          │
│ 5         ┆ 2          │
│ 6         ┆ 5          │
│ 7         ┆ 2          │
│ 8         ┆ 1          │
└───────────┴────────────┘
=== (4) 전환 퍼널 (시간 순/동일 아이템 기준) ===
노출 쌍 수: 643
클릭 쌍 수: 42
좋아요 